### Load data first

In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
import cv2
import time
import sys

sys.path.append(os.path.abspath(".."))   # Add root path to sys.path
os.chdir("..")  # Change working directory to root path

from tqdm import tqdm
from datetime import datetime

### Setup Parameters

In [30]:
DATASET = "KDVN"
MODEL_NAME = "ours"

MAX_RADIUS = 120
NUM_RADII = 4
NUM_SECTORS = 8
DENSITY = 0.05

MAX_VELOCITY = 100
SHAPE_DIFF_WEIGHT = 0.4         # weights of the shape vector difference in the cost function, this will sum with spatial distance weight to 1.0

RADII = np.linspace(start=MAX_RADIUS/NUM_RADII, stop=MAX_RADIUS, num=NUM_RADII, dtype=int).tolist()

In [31]:
from src.preprocessing import read_numpy_grid, nexrad_numpy_preprocessing_pipeline

SOURCE_PATH = f"data/numpy_grid/{DATASET}"
img_paths = [os.path.join(SOURCE_PATH, img_name) for img_name in sorted(os.listdir(SOURCE_PATH)) if img_name.endswith('.npy')]

dbz_maps: list[tuple[np.ndarray, datetime]] = []

for path in tqdm(img_paths, desc="Processing images and detecting storms"):
    file_name = path.split("/")[-1].split(".")[0]

    time_frame = datetime.strptime(file_name[4:19], "%Y%m%d_%H%M%S")
    img = read_numpy_grid(path)
    dbz_maps.append((img, time_frame))

print(f"Number of frames: {len(dbz_maps)} | Shape of each frame: {dbz_maps[0][0].shape}")

Processing images and detecting storms:   0%|          | 0/78 [00:00<?, ?it/s]

Processing images and detecting storms: 100%|██████████| 78/78 [00:00<00:00, 333.88it/s]

Number of frames: 78 | Shape of each frame: (901, 901)


## Load model

In [32]:
from tqdm import tqdm

from src.models import OursPrecipitationModel
from src.identification import HypothesisIdentifier, MorphContourIdentifier

model = OursPrecipitationModel(identifier=MorphContourIdentifier(), radii=RADII, num_sectors=NUM_SECTORS, density=DENSITY)
storms_maps = []

prev_storms = model.identify_storms(dbz_maps[0][0], dbz_maps[0][1], map_id="time_0", threshold=35, filter_area=50)
curr_storms = model.identify_storms(dbz_maps[1][0], dbz_maps[1][1], map_id="time_1", threshold=35, filter_area=50)

## Testing the disparity matrix

In [33]:
from src.models.ours import ParticleMatcher, Particle

prev_storms_list = list(prev_storms.storms)
curr_storms_list = list(curr_storms.storms)

particles_prev: list[Particle] = [Particle(feature=v, storm_id=storm.id) for storm in prev_storms_list\
                    for v in storm.shape_vectors]
particles_curr: list[Particle] = [Particle(feature=v, storm_id=storm.id) for storm in curr_storms_list\
                    for v in storm.shape_vectors]

shape_diff_matrix, displacement_matrix = ParticleMatcher()._construct_disparity_matrix(particles_prev, particles_curr, weights=(0,1))
cost_matrix = SHAPE_DIFF_WEIGHT * shape_diff_matrix + (1 - SHAPE_DIFF_WEIGHT) * displacement_matrix

# Masking displacement matrix where distance is too large
cost_matrix = cost_matrix + (displacement_matrix > MAX_VELOCITY).astype(np.float64) * 3000      # add penalty to those violated
row_ind, col_ind = ParticleMatcher()._hungarian_matching(cost_matrix)

shape_diff_lsts = [shape_diff_matrix[row_ind[i], col_ind[i]] for i in range(len(row_ind))]
displacement_lsts = [displacement_matrix[row_ind[i], col_ind[i]] for i in range(len(row_ind))]

### Matching storms

In [34]:
for s, d in zip(shape_diff_lsts, displacement_lsts):
    print(f"Shape diff: {s:.4f} | Displacement: {d:.2f}")

Shape diff: 17.1756 | Displacement: 6.40
Shape diff: 11.7047 | Displacement: 3.61
Shape diff: 15.9374 | Displacement: 5.39
Shape diff: 16.8226 | Displacement: 5.00
Shape diff: 16.1864 | Displacement: 2.24
Shape diff: 17.5214 | Displacement: 5.39
Shape diff: 15.1658 | Displacement: 5.39
Shape diff: 15.5885 | Displacement: 5.39
Shape diff: 14.6969 | Displacement: 1.00
Shape diff: 19.8494 | Displacement: 5.10
Shape diff: 15.8430 | Displacement: 3.16
Shape diff: 11.6190 | Displacement: 3.16
Shape diff: 9.7980 | Displacement: 4.00
Shape diff: 17.4069 | Displacement: 2.00
Shape diff: 16.8226 | Displacement: 1.00
Shape diff: 11.0905 | Displacement: 3.61
Shape diff: 11.2694 | Displacement: 2.83
Shape diff: 10.7703 | Displacement: 3.61
Shape diff: 8.3666 | Displacement: 5.10
Shape diff: 12.2066 | Displacement: 1.00
Shape diff: 17.6068 | Displacement: 7.07
Shape diff: 10.5357 | Displacement: 4.12
Shape diff: 38.0657 | Displacement: 474.77
Shape diff: 10.8167 | Displacement: 4.12
Shape diff: 10.4